# GNN-IDS: end-to-end pipeline (Google Colab)
Runtime: T4 GPU. Place the repository in Google Drive and set `PROJECT` below. Prepared datasets are read from `data/`, results are written to `results/`.

**Complete run:** with an empty `results/` folder every section computes and prints its metrics (training logs, one metric line per run). Run the notebook from top to bottom.
**Resuming:** if the session disconnects, reconnect and run the notebook again; finished runs print `skip (done)` and only the missing runs are computed. `--force` recomputes a run.
The tables and figures of section 8 are built from all result files in `results/`.

## 0. Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/GNN-IDS (2)/GNN-IDS'     # path of this repository in Drive
%cd {PROJECT}
!pip install -q xgboost lightgbm pytest
import torch; print('GPU available:', torch.cuda.is_available())
HP = '--hidden 128 --layers 2 --lr 0.002 --lam 0.5 --cap 2048'   # graph-model hyper-parameters (selected on validation PR-AUC)

Mounted at /content/drive
/content/drive/MyDrive/GNN-IDS (2)/GNN-IDS
GPU available: True


## 1. Unit tests

In [ ]:
!python -m pytest -q tests

.....................                                                    [100%]
=============================== warnings summary ===============================
tests/test_core.py::test_metrics_perfect_random_and_degenerate
  /usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
    warnings.warn("y_pred contains classes not in y_true")

tests/test_core.py::test_paper_figures_generate
  /content/drive/MyDrive/GNN-IDS (2)/GNN-IDS/gnnids/paper_figs.py:174: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
    im = ax.imshow(Z, cmap="Blues", vmin=0, vmax=1, aspect="auto"); ax.grid(False)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
21 passed, 2 warnings in 65.69s (0:01:05)


## 2. Data
`data/UNSW` and `data/CIC` come from `python -m gnnids prep` (see README). The class-chronological split is derived from the prepared chronological file:

In [ ]:
!python -m gnnids resplit --data data/CIC --out data/CICc --mode classchrono

{
 "rows_raw": 1025000,
 "rows_after_dedup": 982347,
 "rows_used": 949060,
 "split_mode": "classchrono",
 "split_sizes": [
  581057,
  183999,
  184004
 ],
 "attack_rate_by_split": [
  0.12271601581256228,
  0.12270175381387942,
  0.12272015825742918
 ],
 "test_rows_with_identical_features_in_train": 0.16196387035064455,
 "val_rows_with_identical_features_in_train": 0.2134902907081017,
 "purge_ms": 300000,
 "families": [
  "Benign",
  "Bot",
  "Brute_Force_-Web",
  "Brute_Force_-XSS",
  "DDOS_attack-HOIC",
  "DDoS_attacks-LOIC-HTTP",
  "DoS_attacks-GoldenEye",
  "DoS_attacks-Hulk",
  "FTP-BruteForce",
  "Infilteration",
  "SQL_Injection",
  "SSH-Bruteforce"
 ],
 "family_counts": {
  "Benign": 832597,
  "Bot": 9763,
  "Brute_Force_-Web": 65,
  "Brute_Force_-XSS": 2,
  "DDOS_attack-HOIC": 62750,
  "DDoS_attacks-LOIC-HTTP": 13011,
  "DoS_attacks-GoldenEye": 6012,
  "DoS_attacks-Hulk": 2394,
  "FTP-BruteForce": 12788,
  "Infilteration": 4083,
  "SQL_Injection": 20,
  "SSH-Bruteforce": 5575

## 3. Diagnostics: split audit, single-feature shortcuts, sanity checks

In [ ]:
import json
for d in ['UNSW','CIC','CICc']:
    a = json.load(open(f'data/{d}.json'))['audit']
    print(d, a.get('split_mode','chrono'), '| attack rate per split', [round(x,4) for x in a['attack_rate_by_split']],
          '| test rows duplicated in train', round(a['test_rows_with_identical_features_in_train'],3))
    for f, c in a.get('family_counts_by_split', {}).items():
        if f != 'Benign': print('   ', f, c)

UNSW chrono | attack rate per split [0.0208, 0.0694, 0.0673] | test rows duplicated in train 0.324
CIC chrono | attack rate per split [0.1776, 0.0136, 0.061] | test rows duplicated in train 0.13
CICc classchrono | attack rate per split [0.1227, 0.1227, 0.1227] | test rows duplicated in train 0.162
    Bot [5976, 1893, 1894]
    Brute_Force_-Web [40, 12, 13]
    Brute_Force_-XSS [2, 0, 0]
    DDOS_attack-HOIC [38418, 12166, 12166]
    DDoS_attacks-LOIC-HTTP [7965, 2523, 2523]
    DoS_attacks-GoldenEye [3680, 1166, 1166]
    DoS_attacks-Hulk [1465, 464, 465]
    FTP-BruteForce [7828, 2480, 2480]
    Infilteration [2499, 792, 792]
    SQL_Injection [20, 0, 0]
    SSH-Bruteforce [3412, 1081, 1082]


In [ ]:
!python -m gnnids shortcuts --data data/UNSW
!python -m gnnids shortcuts --data data/CICc
!python -m gnnids sanity --data data/UNSW
!python -m gnnids sanity --data data/CICc

                  feature  roc_auc   pr_auc
                  MAX_TTL 1.000000 1.000000
                  MIN_TTL 1.000000 1.000000
           MIN_IP_PKT_LEN 0.999496 0.996992
        SHORTEST_FLOW_PKT 0.999226 0.996006
          TCP_WIN_MAX_OUT 0.993818 0.977913
           TCP_WIN_MAX_IN 0.983675 0.935281
         SERVER_TCP_FLAGS 0.980834 0.710577
  SRC_TO_DST_SECOND_BYTES 0.953336 0.616064
DST_TO_SRC_AVG_THROUGHPUT 0.952343 0.619101
       DST_TO_SRC_IAT_AVG 0.934236 0.540977
saved results/shortcuts/UNSW.csv
                 feature  roc_auc   pr_auc
        LONGEST_FLOW_PKT 0.978707 0.918963
          MAX_IP_PKT_LEN 0.978707 0.918963
               OUT_BYTES 0.955199 0.864706
         TCP_WIN_MAX_OUT 0.954325 0.831831
                L7_PROTO 0.951186 0.639156
          TCP_WIN_MAX_IN 0.939736 0.809450
                IN_BYTES 0.935892 0.723979
NUM_PKTS_UP_TO_128_BYTES 0.932461 0.684354
                OUT_PKTS 0.918407 0.576452
        SERVER_TCP_FLAGS 0.916588 0.515158
saved resu

## 4. Tabular baselines and transfer

In [ ]:
SEEDS_B = [0, 1, 2]
PAIRS = [('UNSW','CICc'), ('CICc','UNSW'), ('CIC','UNSW')]
for s in SEEDS_B:
    for d, t in PAIRS:
        !python -m gnnids baselines --data data/{d} --target data/{t} --seeds {s} --trials 3 --models xgb lgbm topo iponly

UNSW_xgb_s0 {'f1_macro': 0.9997, 'pr_auc': 0.9998, 'tpr_at_1pct_fpr': 1.0}
UNSW_lgbm_s0 {'f1_macro': 0.9997, 'pr_auc': 0.9999, 'tpr_at_1pct_fpr': 1.0}
UNSW_topo_s0 {'f1_macro': 0.9997, 'pr_auc': 0.9998, 'tpr_at_1pct_fpr': 1.0}
UNSW_iponly_s0 {'f1_macro': 1.0, 'pr_auc': 1.0, 'tpr_at_1pct_fpr': 1.0}
CICc_xgb_s0 {'f1_macro': 0.9892, 'pr_auc': 0.9803, 'tpr_at_1pct_fpr': 0.9658}
CICc_lgbm_s0 {'f1_macro': 0.9891, 'pr_auc': 0.98, 'tpr_at_1pct_fpr': 0.9659}
CICc_topo_s0 {'f1_macro': 0.9894, 'pr_auc': 0.9807, 'tpr_at_1pct_fpr': 0.9664}
CICc_iponly_s0 {'f1_macro': 0.9897, 'pr_auc': 0.9991, 'tpr_at_1pct_fpr': 0.9989}
CIC_xgb_s0 {'f1_macro': 0.4933, 'pr_auc': 0.6794, 'tpr_at_1pct_fpr': 0.8614}
CIC_lgbm_s0 {'f1_macro': 0.8726, 'pr_auc': 0.7596, 'tpr_at_1pct_fpr': 0.742}
CIC_topo_s0 {'f1_macro': 0.8364, 'pr_auc': 0.8099, 'tpr_at_1pct_fpr': 0.8321}
CIC_iponly_s0 {'f1_macro': 0.4843, 'pr_auc': 0.0411, 'tpr_at_1pct_fpr': 0.0021}
UNSW_xgb_s1 {'f1_macro': 0.9997, 'pr_auc': 0.9999, 'tpr_at_1pct_fpr': 1.0}

Baseline summary per dataset (mean and sample standard deviation over seeds; paired bootstrap of XGBoost against the other baselines):

In [ ]:
for d in ['UNSW','CICc','CIC']:
    !python -m gnnids summarize --results results --dataset {d} --graph xgb --out report


UNSW: metrics over seeds (std = sample standard deviation)
        n_seeds  pr_auc_mean  pr_auc_std  pr_auc_min  tpr1_mean  acc_mean  bal_acc_mean  f1_mean
model                                                                                           
iponly        3       1.0000      0.0000      1.0000        1.0    1.0000        1.0000   1.0000
lgbm          3       0.9998      0.0001      0.9997        1.0    0.9999        0.9999   0.9997
topo          3       0.9998      0.0000      0.9998        1.0    0.9999        1.0000   0.9997
xgb           3       0.9997      0.0001      0.9996        1.0    0.9999        1.0000   0.9997

UNSW: recall per attack family
family  Analysis  Backdoor  DoS  Exploits  Fuzzers  Generic  Reconnaissance  Shellcode  Worms
model                                                                                        
iponly       1.0       1.0  1.0       1.0      1.0      1.0             1.0        1.0    1.0
lgbm         1.0       1.0  1.0       1.0   

## 5. Graph models: static / temporal E-GraphSAGE, degree features, adversarial training

In [ ]:
SEEDS_T = [0, 1, 2]
for s in SEEDS_T:
    for d, t in [('UNSW','CICc'), ('CICc','UNSW')]:
        !python -m gnnids tegs --data data/{d} --target data/{t} --nomem                         --robust --seeds {s} {HP}
        !python -m gnnids tegs --data data/{d} --target data/{t} --nomem --node_feats            --robust --seeds {s} {HP}
        !python -m gnnids tegs --data data/{d} --target data/{t}                                 --robust --seeds {s} {HP}
        !python -m gnnids tegs --data data/{d} --target data/{t} --node_feats                    --robust --seeds {s} {HP}
        !python -m gnnids tegs --data data/{d} --target data/{t} --node_feats --adv              --robust --latency_caps 1 32 256 --seeds {s} {HP}
        !python -m gnnids tegs --data data/{d} --target data/{t} --nomem --node_feats --adv      --robust --seeds {s} {HP}

267 training batches per epoch, 154 validation, 153 test (cap=2048); few batches = few optimizer steps = under-training
seed 0 epoch 0 loss 0.0005 val PR-AUC 0.9988
seed 0 epoch 1 loss 0.0015 val PR-AUC 0.9912
seed 0 epoch 2 loss 0.0005 val PR-AUC 0.9981
seed 0 epoch 3 loss 0.0005 val PR-AUC 0.9990
seed 0 epoch 4 loss 0.0000 val PR-AUC 0.9990
seed 0 epoch 5 loss 0.0000 val PR-AUC 0.9990
seed 0 epoch 6 loss 0.0000 val PR-AUC 0.9981
seed 0 epoch 7 loss 0.0000 val PR-AUC 0.9991
seed 0 epoch 8 loss 0.0000 val PR-AUC 0.9987
seed 0 epoch 9 loss 0.0000 val PR-AUC 0.9989
seed 0 epoch 10 loss 0.0000 val PR-AUC 0.9991
seed 0 epoch 11 loss 0.0000 val PR-AUC 0.9991
seed 0 epoch 12 loss 0.0000 val PR-AUC 0.9991
seed 0 epoch 13 loss 0.0000 val PR-AUC 0.9991
UNSW_tegs_nomem_s0 {'f1_macro': 0.9997, 'pr_auc': 0.9992, 'tpr_at_1pct_fpr': 1.0}
267 training batches per epoch, 154 validation, 153 test (cap=2048); few batches = few optimizer steps = under-training
seed 0 epoch 0 loss 0.0009 val PR-AUC 0.9976

## 6. Zero-day evaluation on the chronological CIC-2018 split

In [ ]:
for s in SEEDS_T:
    !python -m gnnids baselines --data data/CIC --target data/UNSW --seeds {s} --trials 3 --models xgb lgbm topo iponly
    !python -m gnnids tegs --data data/CIC --target data/UNSW --nomem --node_feats --seeds {s} {HP}
    !python -m gnnids tegs --data data/CIC --target data/UNSW --node_feats --seeds {s} {HP}
    !python -m gnnids tegs --data data/CIC --target data/UNSW --nomem --seeds {s} {HP}

skip (done): CIC_xgb_s0
skip (done): CIC_lgbm_s0
skip (done): CIC_topo_s0
skip (done): CIC_iponly_s0
376 training batches per epoch, 127 validation, 120 test (cap=2048); few batches = few optimizer steps = under-training
seed 0 epoch 0 loss 0.0112 val PR-AUC 0.0358
seed 0 epoch 1 loss 0.0002 val PR-AUC 0.0493
seed 0 epoch 2 loss 0.0004 val PR-AUC 0.0405
seed 0 epoch 3 loss 0.0001 val PR-AUC 0.0392
seed 0 epoch 4 loss 0.0001 val PR-AUC 0.0473
seed 0 epoch 5 loss 0.0731 val PR-AUC 0.0176
seed 0 epoch 6 loss 0.0002 val PR-AUC 0.0179
seed 0 epoch 7 loss 0.0001 val PR-AUC 0.0194
CIC_tegs_nomem_nf_s0 {'f1_macro': 0.9569, 'pr_auc': 0.8765, 'tpr_at_1pct_fpr': 0.8628}
376 training batches per epoch, 127 validation, 120 test (cap=2048); few batches = few optimizer steps = under-training
seed 0 epoch 0 loss 0.1441 val PR-AUC 0.0125
seed 0 epoch 1 loss 0.0269 val PR-AUC 0.0110
seed 0 epoch 2 loss 0.0215 val PR-AUC 0.0128
seed 0 epoch 3 loss 0.0091 val PR-AUC 0.0114
seed 0 epoch 4 loss 0.0192 val P

## 7. Window length and leave-one-family-out

In [ ]:
for d in ['UNSW','CICc']:
    for dl in [30, 120, 300]:
        !python -m gnnids tegs --data data/{d} --delta {dl} --tag _d{dl} --node_feats --seeds 0 1 2 {HP}
for d, fams in [('UNSW', ['Exploits','Fuzzers','Reconnaissance']), ('CICc', ['Bot','Infilteration','DDOS_attack-HOIC'])]:
    for f in fams:
        !python -m gnnids baselines --data data/{d} --models xgb lgbm --seeds 0 --holdout {f} --trials 3
        !python -m gnnids tegs --data data/{d} --seeds 0 --nomem --node_feats --holdout {f} {HP}
        !python -m gnnids tegs --data data/{d} --seeds 0 --node_feats --holdout {f} {HP}

506 training batches per epoch, 263 validation, 261 test (cap=2048); few batches = few optimizer steps = under-training
seed 0 epoch 0 loss 0.0017 val PR-AUC 0.9986
seed 0 epoch 1 loss 0.0004 val PR-AUC 0.9984
seed 0 epoch 2 loss 0.0004 val PR-AUC 0.9990
seed 0 epoch 3 loss 0.0012 val PR-AUC 0.9991
seed 0 epoch 4 loss 0.0018 val PR-AUC 0.9983
seed 0 epoch 5 loss 0.0005 val PR-AUC 0.9990
seed 0 epoch 6 loss 0.0021 val PR-AUC 0.9987
seed 0 epoch 7 loss 0.0014 val PR-AUC 0.9989
seed 0 epoch 8 loss 0.0002 val PR-AUC 0.9988
seed 0 epoch 9 loss 0.0000 val PR-AUC 0.9990
UNSW_tegs_nf_d30_s0 {'f1_macro': 0.9997, 'pr_auc': 0.9987, 'tpr_at_1pct_fpr': 1.0}
seed 1 epoch 0 loss 0.0018 val PR-AUC 0.9981
seed 1 epoch 1 loss 0.0005 val PR-AUC 0.9992
seed 1 epoch 2 loss 0.0007 val PR-AUC 0.9986
seed 1 epoch 3 loss 0.0005 val PR-AUC 0.9975
seed 1 epoch 4 loss 0.0002 val PR-AUC 0.9980
seed 1 epoch 5 loss 0.0025 val PR-AUC 0.9989
seed 1 epoch 6 loss 0.0002 val PR-AUC 0.9987
seed 1 epoch 7 loss 0.0004 val P

## 8. Tables, significance tests and figures

In [ ]:
!python -m gnnids summarize --results results --dataset CIC  --graph tegs_nomem_nf --out report
!python -m gnnids summarize --results results --dataset CICc --graph tegs_nf --out report
!python -m gnnids stats  --results results --out report
!python -m gnnids report --results results --data_dir data --out report
!python -m gnnids paper  --results results --data_dir data --out report/paper


CIC: metrics over seeds (std = sample standard deviation)
               n_seeds  pr_auc_mean  pr_auc_std  pr_auc_min  tpr1_mean  acc_mean  bal_acc_mean  f1_mean
model                                                                                                  
tegs_nomem_nf        3       0.8776      0.0016      0.8765     0.8645    0.9907        0.9299   0.9567
tegs_nomem           3       0.8560      0.0167      0.8425     0.8654    0.9835        0.9281   0.9297
topo                 3       0.7679      0.0959      0.6582     0.7449    0.9657        0.7275   0.7553
tegs_nf              3       0.6599      0.3782      0.2232     0.5712    0.9664        0.7812   0.7774
xgb                  3       0.5125      0.2003      0.2904     0.2918    0.9330        0.5006   0.4899
lgbm                 3       0.4668      0.2544      0.3000     0.2517    0.9441        0.6111   0.6157
iponly               3       0.0416      0.0016      0.0404     0.0019    0.9387        0.4999   0.4843

CIC:

In [ ]:
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('report/paper/*.png')):
    print(f); display(Image(f, width=900))

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
print(pd.read_csv('report/summary.csv').round(4).to_string())

   Unnamed: 0         Unnamed: 1                mean               mean.1               mean.2               mean.3              mean.4                 mean.5                 mean.6                  mean.7                mean.8                mean.9              mean.10                     std                   std.1                   std.2                   std.3                   std.4                   std.5                   std.6                   std.7                   std.8                   std.9                  std.10 n_seeds
0         NaN                NaN            accuracy    balanced_accuracy               pr_auc              roc_auc            f1_macro                    mcc        tpr_at_1pct_fpr            fpr_at_95tpr              c_pr_auc     c_tpr_at_1pct_fpr           c_f1_macro                accuracy       balanced_accuracy                  pr_auc                 roc_auc                f1_macro                     mcc         tpr_at_1pct_fpr            fpr_at_

In [3]:
HP = '--hidden 128 --layers 2 --lr 0.002 --lam 0.5 --cap 2048'
for d in ['UNSW', 'CICc', 'CIC']:
    # topology-only: flow features hidden, graph structure + degree features kept
    !python -m gnnids tegs --data data/{d} --nomem --node_feats --no_edge --tag _noedge --seeds 0 1 2 {HP}
    # depth sweep (2 layers is already done)
    !python -m gnnids tegs --data data/{d} --nomem --node_feats --tag _L1 --seeds 0 1 2 {HP} --layers 1
    !python -m gnnids tegs --data data/{d} --nomem --node_feats --tag _L3 --seeds 0 1 2 {HP} --layers 3
!python -m gnnids summarize --results results --out report

267 training batches per epoch, 154 validation, 153 test (cap=2048); few batches = few optimizer steps = under-training
seed 0 epoch 0 loss 0.0071 val PR-AUC 0.4354
seed 0 epoch 1 loss 0.0060 val PR-AUC 0.4813
seed 0 epoch 2 loss 0.0060 val PR-AUC 0.4844
seed 0 epoch 3 loss 0.0046 val PR-AUC 0.4555
seed 0 epoch 4 loss 0.0052 val PR-AUC 0.5250
seed 0 epoch 5 loss 0.0039 val PR-AUC 0.5760
seed 0 epoch 6 loss 0.0041 val PR-AUC 0.5355
seed 0 epoch 7 loss 0.0041 val PR-AUC 0.5811
seed 0 epoch 8 loss 0.0040 val PR-AUC 0.6125
seed 0 epoch 9 loss 0.0046 val PR-AUC 0.6087
seed 0 epoch 10 loss 0.0033 val PR-AUC 0.6868
seed 0 epoch 11 loss 0.0030 val PR-AUC 0.6983
seed 0 epoch 12 loss 0.0036 val PR-AUC 0.5402
seed 0 epoch 13 loss 0.0035 val PR-AUC 0.6336
seed 0 epoch 14 loss 0.0025 val PR-AUC 0.6697
seed 0 epoch 15 loss 0.0033 val PR-AUC 0.7138
seed 0 epoch 16 loss 0.0034 val PR-AUC 0.7189
seed 0 epoch 17 loss 0.0026 val PR-AUC 0.8373
seed 0 epoch 18 loss 0.0032 val PR-AUC 0.6593
seed 0 epoch 19 